# 02 – Collect EC Legacy Formal Decisions

## Zweck dieses Notebooks

Dieses Notebook sammelt ältere EC-Antitrust-Fälle aus den offiziellen Legacy-Seiten der Europäischen Kommission.

**Bewusst enger Fokus:**
- Nur `Cases closed by Formal Decisions 81/82 (ex 85/86)` aus den Legacy-Jahresseiten
- Direkte Abfrage der bekannten Jahresseiten-URLs (keine breite Link-Erkennung)
- Enger, strukturspezifischer Parser – orientiert an der alten, funktionierenden Extraktionslogik
- Kein EUR-Lex, keine Comfort Letters, keine Anreicherung, kein Merge



## 1. Imports und Konfiguration


In [33]:
import re
import time
import requests
import pandas as pd
from pathlib import Path
from bs4 import BeautifulSoup

print("Imports OK")


Imports OK


## 2. Konstanten


In [34]:
# Basis-URL für die Legacy-Jahresseiten
BASE_URL = "https://ec.europa.eu/competition/antitrust/closed/en/"

# Ausgabeverzeichnisse
OUTPUT_DIR = Path("data/processed")
RAW_DIR    = Path("data/raw/ec_legacy_formal_decisions")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RAW_DIR.mkdir(parents=True, exist_ok=True)

# Jahresbereich der Legacy-Seiten
YEAR_START = 1964
YEAR_END   = 1997  # inklusiv

# EUR-Lex Sprache für document_url (variabel: "EN", "DE", "FR", ...)
EURLEX_LANG = "EN"

print(f"BASE_URL   : {BASE_URL}")
print(f"OUTPUT_DIR : {OUTPUT_DIR}")
print(f"RAW_DIR    : {RAW_DIR}")
print(f"Jahresbereich: {YEAR_START}–{YEAR_END}")
print(f"EURLEX_LANG: {EURLEX_LANG}")


BASE_URL   : https://ec.europa.eu/competition/antitrust/closed/en/
OUTPUT_DIR : data\processed
RAW_DIR    : data\raw\ec_legacy_formal_decisions
Jahresbereich: 1964–1997
EURLEX_LANG: EN


## 3. Session / Request-Helfer


In [35]:
def make_session() -> requests.Session:
    """Erstellt eine requests.Session mit sinnvollen Headern."""
    s = requests.Session()
    s.headers.update({
        "User-Agent": "Mozilla/5.0 (compatible; EC-Antitrust-Scraper/2.0; +research)",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.5",
    })
    return s


def fetch_html(session: requests.Session, url: str, timeout: int = 20):
    """
    Lädt eine URL und gibt (status_code, text) zurück.
    Bei Fehler: (status_code_oder_None, None).
    """
    try:
        r = session.get(url, timeout=timeout)
        if r.status_code == 404:
            return 404, None
        r.raise_for_status()
        r.encoding = r.apparent_encoding or "ISO-8859-1"
        return r.status_code, r.text
    except requests.exceptions.Timeout:
        return None, None
    except requests.exceptions.RequestException:
        return None, None


def save_raw_html(path: Path, text: str) -> None:
    """Speichert rohen HTML-Text in eine Datei."""
    path.write_text(text, encoding="utf-8", errors="replace")


print("Helfer-Funktionen definiert.")


Helfer-Funktionen definiert.


## 4. Jahresseiten-Liste erzeugen

Die URLs werden direkt und kontrolliert konstruiert – keine heuristische Link-Erkennung.


In [36]:
year_pages = []
for year in range(YEAR_START, YEAR_END + 1):
    url = f"{BASE_URL}{year}.html"
    year_pages.append({"year": year, "year_url": url})

df_years = pd.DataFrame(year_pages)
print(f"Geplante Jahresseiten: {len(df_years)}")
df_years.head(10)


Geplante Jahresseiten: 34


,year,year_url
0,1964,https://ec.europa.eu/competition/antitrust/clo...
1,1965,https://ec.europa.eu/competition/antitrust/clo...
2,1966,https://ec.europa.eu/competition/antitrust/clo...
3,1967,https://ec.europa.eu/competition/antitrust/clo...
4,1968,https://ec.europa.eu/competition/antitrust/clo...
5,1969,https://ec.europa.eu/competition/antitrust/clo...
6,1970,https://ec.europa.eu/competition/antitrust/clo...
7,1971,https://ec.europa.eu/competition/antitrust/clo...
8,1972,https://ec.europa.eu/competition/antitrust/clo...
9,1973,https://ec.europa.eu/competition/antitrust/clo...


## 5. CELEX-Hilfsfunktionen


In [37]:
def _normalize_celex(s: str) -> str:
    """Entfernt Whitespace, behält Großschreibung."""
    return re.sub(r"\s+", "", s or "")


def _to_celex_complete(celex_no: str) -> str:
    """
    Expandiert 3+YY+L+NNNN (z.B. 371D0023) zu 3+YYYY+L+NNNN (31971D0023).
    Bereits vollständige 10-stellige CELEX-Nummern werden unverändert zurückgegeben.
    """
    c = _normalize_celex(celex_no)
    if not c:
        return ""
    if re.fullmatch(r"3\d{4}[A-Z]\d{4}", c):
        return c
    m = re.fullmatch(r"3(\d{2})([A-Z])(\d{4})", c)
    if m:
        yy, letter, num = m.groups()
        yyyy = f"19{yy}" if int(yy) >= 50 else f"20{yy}"
        return f"3{yyyy}{letter}{num}"
    return c


print("CELEX-Hilfsfunktionen definiert.")


CELEX-Hilfsfunktionen definiert.


## 6. Enge Extraktionslogik pro Jahresseite

Der Parser orientiert sich direkt an der alten, funktionierenden Logik:
- Falltitel aus `<a name="...">` Ankern
- Metadaten aus dem nächsten `<font>`-Block mit Datum
- Kein generisches Einsammeln aller `<li>` oder `<p>`-Elemente


In [38]:
DATE_PAT = re.compile(r"\b\d{2}\.\d{2}\.\d{4}\b")


def parse_case_entries_from_year_page(soup: BeautifulSoup, year: int, source_url: str, lang: str = EURLEX_LANG) -> list:
    """
    Extrahiert Falleinträge aus einer Legacy-Jahresseite.

    Logik (eng auf die alte Seitenstruktur zugeschnitten):
    - Sucht alle <a name="..."> Anker als Fallstartpunkte
    - Liest den nächsten <font>-Block mit Datum als Detailblock
    - Extrahiert Metadaten direkt aus diesem Block
    """
    entries = []
    anchors = soup.find_all("a", attrs={"name": True})

    for idx, a_tag in enumerate(anchors):
        # Überspringe Navigation-Anker ("top" etc.)
        name_attr = a_tag.get("name", "").lower().strip()
        if name_attr in ("top", "", "bottom", "index"):
            continue

        title = a_tag.get_text(" ", strip=True)
        if not title:
            continue

        # Suche den nächsten <font>-Block mit einem Datum
        details_font = None
        candidate = a_tag.find_next("font")
        steps = 0
        while candidate and steps < 40:
            text = " ".join(candidate.stripped_strings)
            if DATE_PAT.search(text):
                details_font = candidate
                break
            candidate = candidate.find_next("font")
            steps += 1

        if not details_font:
            continue

        details_text = " ".join(details_font.stripped_strings)

        # --- Datum ---
        m_date = DATE_PAT.search(details_text)
        decision_date_raw = m_date.group(0) if m_date else ""

        # --- Official Journal ---
        oj_str = ""
        m_oj = re.search(
            r"Official Journal\s*:\s*(.+?)(?=\s+Celex No\.\s*:|\s+\b[A-Z]{1,3}\s*/\s*\d{1,6}\b|$)",
            details_text
        )
        if m_oj:
            oj_str = m_oj.group(1).strip()

        # --- CELEX (bevorzugt aus href numdoc=, Fallback aus Text) ---
        celex_raw = ""
        celex_display = ""
        document_url = ""

        celex_anchor = details_font.find("a", href=re.compile(r"CELEXnumdoc", re.I))
        if celex_anchor and celex_anchor.has_attr("href"):
            m_href = re.search(r"numdoc=([0-9A-Za-z]+)", celex_anchor["href"])
            if m_href:
                celex_raw = _normalize_celex(m_href.group(1))
            celex_display = celex_anchor.get_text(" ", strip=True)

        if not celex_raw:
            m_celex_text = re.search(
                r"Celex No\.\s*:\s*([0-9A-Za-z\s]+?)(?=\s+-\s+|\s+\b[A-Z]{1,3}\s*/\s*\d{1,6}\b|$)",
                details_text
            )
            if m_celex_text:
                celex_display = m_celex_text.group(1).strip()
                celex_raw = _normalize_celex(celex_display)

        celex_complete = _to_celex_complete(celex_raw)

        # EUR-Lex document_url aus celex_complete + lang konstruieren (DE als Fallback)
        if celex_complete:
            document_url    = f"https://eur-lex.europa.eu/legal-content/{lang}/TXT/HTML/?uri=CELEX:{celex_complete}"
            document_url_de = f"https://eur-lex.europa.eu/legal-content/DE/TXT/HTML/?uri=CELEX:{celex_complete}"
        else:
            document_url    = ""
            document_url_de = ""

        # --- Fallnummern (IV/…) ---
        raw_cases = re.findall(r"\b([A-Z]{1,3})\s*/\s*(\d{1,6})\b", details_text)
        seen = set()
        case_numbers_list = []
        for prefix, num in raw_cases:
            normalized = f"{prefix}/{num}"
            if normalized not in seen:
                seen.add(normalized)
                case_numbers_list.append(normalized)
        case_number_raw = "; ".join(case_numbers_list)

        # --- Entscheidungstyp (nach Datum, vor OJ/CELEX) ---
        decision_type = ""
        if decision_date_raw:
            post_date = details_text.split(decision_date_raw, 1)[1].strip()
            cut_points = []
            for marker in [
                r"Official Journal\s*:",
                r"Celex No\.\s*:",
                r"\b[A-Z]{1,3}\s*/\s*\d{1,6}\b"
            ]:
                m = re.search(marker, post_date)
                if m:
                    cut_points.append(m.start())
            end_idx = min(cut_points) if cut_points else len(post_date)
            decision_type = post_date[:end_idx].strip(" -\u00A0").strip()

        # --- Published / Not Published ---
        details_lower = details_text.lower()
        if "not published" in details_lower:
            published_flag = False
            official_journal_flag = False
        elif oj_str or re.search(r"OJ\s+[LC]\s*\d", details_text):
            published_flag = True
            official_journal_flag = True
        elif celex_raw:
            published_flag = True
            official_journal_flag = False
        else:
            published_flag = "Unknown"
            official_journal_flag = "Unknown"

        # --- Rechtliche Grundlage (optional) ---
        legal_basis_raw = ""
        m_legal = re.search(r"Art(?:icle|\.)?\s*(\d+[a-z]?)\b", details_text, re.I)
        if m_legal:
            legal_basis_raw = m_legal.group(0).strip()

        # --- Notizen (optional) ---
        notes_raw = ""
        m_notes = re.search(r"Note\s*:\s*(.+?)(?=\s+[A-Z]|$)", details_text)
        if m_notes:
            notes_raw = m_notes.group(1).strip()

        # --- Record-ID ---
        legacy_record_id = f"{year}_{idx:04d}"

        entries.append({
            "legacy_record_id":      legacy_record_id,
            "year_page":             year,
            "source_url":            source_url,
            "case_title":            title,
            "case_detail_text_raw":  details_text,
            "decision_date_raw":     decision_date_raw,
            "decision_type_raw":     decision_type,
            "publication_ref_raw":   oj_str,
            "case_number_raw":       case_number_raw,
            "celex_raw":             celex_raw,
            "celex_display":         celex_display,
            "celex_complete":        celex_complete,
            "document_url":          document_url,
            "document_url_de":       document_url_de,
            "published_flag":        published_flag,
            "official_journal_flag": official_journal_flag,
            "legal_basis_raw":       legal_basis_raw,
            "notes_raw":             notes_raw,
        })

    return entries


print("Parser-Funktion definiert.")


Parser-Funktion definiert.


## 7. Jahresseiten laden und parsen


In [39]:
session = make_session()

all_records = []
success_years = []
failed_years  = []

for row in df_years.itertuples():
    year     = row.year
    year_url = row.year_url

    status_code, html = fetch_html(session, year_url)

    if html is None:
        reason = "404" if status_code == 404 else (f"HTTP {status_code}" if status_code else "Timeout/Error")
        print(f"   ⚠️  {year}: {reason} – {year_url}")
        failed_years.append({"year": year, "url": year_url, "reason": reason})
        continue

    # Optional: rohe HTML speichern
    raw_path = RAW_DIR / f"formal_decision_{year}.html"
    save_raw_html(raw_path, html)

    soup    = BeautifulSoup(html, "html.parser")
    entries = parse_case_entries_from_year_page(soup, year, year_url, lang=EURLEX_LANG)

    all_records.extend(entries)
    success_years.append(year)
    print(f"   ✅ {year}: {len(entries)} Fälle extrahiert")

    time.sleep(0.3)  # höfliche Pause

print(f"\nFertig. Erfolgreich: {len(success_years)}, Fehlgeschlagen: {len(failed_years)}")
print(f"Extrahierte Fälle gesamt: {len(all_records)}")


   ✅ 1964: 5 Fälle extrahiert
   ✅ 1965: 3 Fälle extrahiert
   ✅ 1966: 0 Fälle extrahiert
   ✅ 1967: 1 Fälle extrahiert
   ✅ 1968: 8 Fälle extrahiert
   ✅ 1969: 10 Fälle extrahiert
   ✅ 1970: 7 Fälle extrahiert
   ✅ 1971: 18 Fälle extrahiert
   ✅ 1972: 15 Fälle extrahiert
   ✅ 1973: 8 Fälle extrahiert
   ✅ 1974: 11 Fälle extrahiert
   ✅ 1975: 16 Fälle extrahiert
   ✅ 1976: 8 Fälle extrahiert
   ✅ 1977: 19 Fälle extrahiert
   ✅ 1978: 12 Fälle extrahiert
   ✅ 1979: 12 Fälle extrahiert
   ✅ 1980: 9 Fälle extrahiert
   ✅ 1981: 15 Fälle extrahiert
   ✅ 1982: 13 Fälle extrahiert
   ✅ 1983: 17 Fälle extrahiert
   ✅ 1984: 20 Fälle extrahiert
   ✅ 1985: 17 Fälle extrahiert
   ✅ 1986: 21 Fälle extrahiert
   ✅ 1987: 15 Fälle extrahiert
   ✅ 1988: 24 Fälle extrahiert
   ✅ 1989: 14 Fälle extrahiert
   ✅ 1990: 16 Fälle extrahiert
   ✅ 1991: 16 Fälle extrahiert
   ✅ 1992: 32 Fälle extrahiert
   ✅ 1993: 0 Fälle extrahiert
   ✅ 1994: 0 Fälle extrahiert
   ✅ 1995: 12 Fälle extrahiert
   ✅ 1996: 22 Fälle

## 8. DataFrame bauen


In [40]:
df = pd.DataFrame(all_records)

# Kontrollspalten hinzufügen
df["extraction_year"] = 2024
df["source_type"]     = "formal_decision"
df["parse_success"]   = df["case_title"].notna() & (df["case_title"] != "")

print(f"DataFrame: {len(df)} Zeilen, {len(df.columns)} Spalten")
print(f"Spalten: {list(df.columns)}")
df.head(3)


DataFrame: 437 Zeilen, 20 Spalten
Spalten: ['legacy_record_id', 'year_page', 'source_url', 'case_title', 'case_detail_text_raw', 'decision_date_raw', 'decision_type_raw', 'publication_ref_raw', 'case_number_raw', 'celex_raw', 'celex_display', 'celex_complete', 'document_url', 'published_flag', 'official_journal_flag', 'legal_basis_raw', 'notes_raw', 'extraction_year', 'source_type', 'parse_success']


,legacy_record_id,year_page,source_url,case_title,case_detail_text_raw,decision_date_raw,decision_type_raw,publication_ref_raw,case_number_raw,celex_raw,celex_display,celex_complete,document_url,published_flag,official_journal_flag,legal_basis_raw,notes_raw,extraction_year,source_type,parse_success
0,1964_0001,1964,https://ec.europa.eu/competition/antitrust/clo...,Deca,22.10.1964 \n\nNegative clearance Art.81(1) ...,22.10.1964,Negative clearance Art.81(1) [ex 85(1)],L - 31/10/1964 Page : 2761,IV/71,364D0599,364D05\n99,31964D0599,https://eur-lex.europa.eu/legal-content/EN/TXT...,True,True,Art.81,,2024,formal_decision,True
1,1964_0002,1964,https://ec.europa.eu/competition/antitrust/clo...,Grundig-Consten,23.09.1964 \n\nInfringement Art.81 [ex 85] O...,23.09.1964,Infringement Art.81 [ex 85],L - 20/10/1964 Page : 2545,IV/3344; IV/4,364D0566,364D05\n66,31964D0566,https://eur-lex.europa.eu/legal-content/EN/TXT...,True,True,Art.81,,2024,formal_decision,True
2,1964_0003,1964,https://ec.europa.eu/competition/antitrust/clo...,Nicholas Freres + Vitapro,30.07.1964 \n\nNegative clearance Art.81(1) ...,30.07.1964,Negative clearance Art.81(1) [ex 85(1)],L - 26/08/1964 Page : 2287,IV/95,364D0502,364D05\n02,31964D0502,https://eur-lex.europa.eu/legal-content/EN/TXT...,True,True,Art.81,,2024,formal_decision,True


## 9. Qualitätschecks


In [41]:
print("=" * 55)
print("QUALITÄTSCHECKS")
print("=" * 55)
print(f"Geplante Jahresseiten       : {len(df_years)}")
print(f"Erfolgreich geladen         : {len(success_years)}")
print(f"Fehlgeschlagen              : {len(failed_years)}")
print(f"Extrahierte Fälle gesamt    : {len(df)}")

if len(df) > 0:
    pub_true  = (df["published_flag"] == True).sum()
    pub_false = (df["published_flag"] == False).sum()
    pub_unk   = (df["published_flag"] == "Unknown").sum()
    has_docurl = (df["document_url"] != "").sum()
    has_celex  = (df["celex_raw"] != "").sum()

    print(f"published_flag = True       : {pub_true}")
    print(f"published_flag = False      : {pub_false}")
    print(f"published_flag = Unknown    : {pub_unk}")
    print(f"Fälle mit Dokumentlink      : {has_docurl}")
    print(f"Fälle mit CELEX-Hinweis     : {has_celex}")

    print("\n--- Fallzahlen pro Jahr ---")
    print(df.groupby("year_page").size().to_string())

if failed_years:
    print("\n--- Fehlgeschlagene Jahre ---")
    for f in failed_years:
        print(f"  {f['year']}: {f['reason']}")


QUALITÄTSCHECKS
Geplante Jahresseiten       : 34
Erfolgreich geladen         : 34
Fehlgeschlagen              : 0
Extrahierte Fälle gesamt    : 437
published_flag = True       : 383
published_flag = False      : 0
published_flag = Unknown    : 54
Fälle mit Dokumentlink      : 383
Fälle mit CELEX-Hinweis     : 383

--- Fallzahlen pro Jahr ---
year_page
1964     5
1965     3
1967     1
1968     8
1969    10
1970     7
1971    18
1972    15
1973     8
1974    11
1975    16
1976     8
1977    19
1978    12
1979    12
1980     9
1981    15
1982    13
1983    17
1984    20
1985    17
1986    21
1987    15
1988    24
1989    14
1990    16
1991    16
1992    32
1995    12
1996    22
1997    21


In [42]:
# Vorschau der ersten Zeilen
if len(df) > 0:
    display(df[["year_page", "case_title", "decision_date_raw", "celex_raw", "published_flag"]].head(10))


,year_page,case_title,decision_date_raw,celex_raw,published_flag
0,1964,Deca,22.10.1964,364D0599,True
1,1964,Grundig-Consten,23.09.1964,364D0566,True
2,1964,Nicholas Freres + Vitapro,30.07.1964,364D0502,True
3,1964,Bendix + Mertens and Straat,01.06.1964,364D0344,True
4,1964,Grosfillex + Fillistorf,11.03.1964,364D0233,True
5,1965,Maison Jalatte,17.12.1965,366D0005,True
6,1965,Hummel - Isbecque,17.09.1965,365D0426,True
7,1965,Dru-Blondel,08.07.1965,365D0366,True
8,1967,Transocean Marine Paint Association,27.06.1967,367D0454,True
9,1968,CFA,06.11.1968,368D0377,True


In [43]:
# Beispiele extrahierter Detailtexte
if len(df) > 0:
    print("--- Beispiele extrahierter Detailtexte ---")
    sample = df[df["case_detail_text_raw"] != ""].head(5)
    for _, row in sample.iterrows():
        print(f"\n[{row['year_page']}] {row['case_title']}")
        print(f"  Detail: {row['case_detail_text_raw'][:200]}")


--- Beispiele extrahierter Detailtexte ---

[1964] Deca
  Detail: 22.10.1964   

Negative clearance Art.81(1) [ex 85(1)] Official Journal :

L - 31/10/1964 Page : 2761   

Celex No. : 364D05
99 - 

IV/71

[1964] Grundig-Consten
  Detail: 23.09.1964   

Infringement Art.81 [ex 85] Official Journal :

L - 20/10/1964 Page : 2545   

Celex No. : 364D05
66 - 

IV/3344

   IV/4

[1964] Nicholas Freres + Vitapro
  Detail: 30.07.1964   

Negative clearance Art.81(1) [ex 85(1)] Official Journal :

L - 26/08/1964 Page : 2287   

Celex No. : 364D05
02 - 

IV/95

[1964] Bendix + Mertens and Straat
  Detail: 01.06.1964   

Negative clearance Art.81(1) [ex 85(1)] Official Journal :

L - 10/06/1964 Page : 1426   
Celex No. : 364D03
44 - 


IV/12868

[1964] Grosfillex + Fillistorf
  Detail: 11.03.1964   

Negative clearance Art.81(1) [ex 85(1)] Official Journal :

L - 9/04/1964 Page : 915   

Celex No. : 364D02
33 - 

IV/61


## 10. Export


In [44]:
if len(df) > 0:
    # CSV (Pflicht)
    csv_path = OUTPUT_DIR / "ec_legacy_formal_decisions.csv"
    df.to_csv(csv_path, index=False, encoding="utf-8")
    print(f"✅ CSV gespeichert → {csv_path}")

    # Parquet (optional)
    try:
        parquet_path = OUTPUT_DIR / "ec_legacy_formal_decisions.parquet"
        df.to_parquet(parquet_path, index=False)
        print(f"✅ Parquet gespeichert → {parquet_path}")
    except Exception as e:
        print(f"ℹ️  Parquet nicht gespeichert: {e}")
else:
    print("⚠️  Keine Daten zum Speichern.")


✅ CSV gespeichert → data\processed\ec_legacy_formal_decisions.csv
ℹ️  Parquet nicht gespeichert: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - `Import pyarrow` failed. pyarrow is required for parquet support. Use pip or conda to install the pyarrow package.
 - `Import fastparquet` failed. fastparquet is required for parquet support. Use pip or conda to install the fastparquet package.
